# Notebook 06 — Train Best Model (Random Forest)

This notebook is **standalone** — it does not require any other notebook to be run first.

It:
1. Loads the raw PJM East dataset
2. Cleans and preprocesses it
3. Creates 24-hour sliding window sequences
4. Trains the Random Forest model (best performing model in this research)
5. Evaluates performance
6. Saves `rf_pjm.pkl`, `pjm_scaler.pkl`, and `pjm_processed.csv`

These saved files are required by the prediction web app (`predict_app/app.py`).

**Expected runtime:** 20–30 minutes

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Paths
BASE   = os.path.join('..', '')
RAW    = os.path.join(BASE, 'data', 'raw')
PROC   = os.path.join(BASE, 'data', 'processed')
MODELS = os.path.join(BASE, 'models')

os.makedirs(PROC,   exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

WINDOW = 24  # 24-hour look-back window

print('Libraries loaded.')
print(f'Raw data folder : {os.path.abspath(RAW)}')
print(f'Models folder   : {os.path.abspath(MODELS)}')

In [ ]:
# ── Load raw PJM East data ────────────────────────────────────────────────────
pjm = pd.read_csv(os.path.join(RAW, 'PJME_hourly.csv'))
print(f'Raw shape: {pjm.shape}')
print(pjm.head(3))

# Parse datetime and sort
pjm['Datetime'] = pd.to_datetime(pjm['Datetime'])
pjm = pjm.sort_values('Datetime').set_index('Datetime')

# Remove duplicate timestamps
dupes = pjm.index.duplicated().sum()
print(f'Duplicate timestamps removed: {dupes}')
pjm = pjm[~pjm.index.duplicated(keep='first')]

# Reindex to complete hourly range (fill any missing hours)
full_idx = pd.date_range(start=pjm.index.min(), end=pjm.index.max(), freq='h')
pjm = pjm.reindex(full_idx).ffill().bfill()
pjm.index.name = 'Datetime'
print(f'After reindex: {pjm.shape}  |  missing values: {pjm.isnull().sum().sum()}')

In [ ]:
# ── Remove outliers using IQR clipping ───────────────────────────────────────
col = 'PJME_MW'
Q1, Q3 = pjm[col].quantile(0.25), pjm[col].quantile(0.75)
IQR = Q3 - Q1
lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
n_outliers = ((pjm[col] < lo) | (pjm[col] > hi)).sum()
print(f'Outliers clipped: {n_outliers}  (range kept: {lo:.0f} – {hi:.0f} MW)')
pjm[col] = pjm[col].clip(lo, hi)

# ── Add time features ────────────────────────────────────────────────────────
def get_season(m):
    if m in [12, 1, 2]:  return 0  # Winter
    if m in [3,  4, 5]:  return 1  # Spring
    if m in [6,  7, 8]:  return 2  # Summer
    return 3                        # Autumn

pjm['hour']         = pjm.index.hour
pjm['day_of_week']  = pjm.index.dayofweek
pjm['day_of_month'] = pjm.index.day
pjm['month']        = pjm.index.month
pjm['year']         = pjm.index.year
pjm['is_weekend']   = (pjm.index.dayofweek >= 5).astype(int)
pjm['season']       = pjm['month'].map(get_season)

# ── Min-Max normalisation ────────────────────────────────────────────────────
scaler = MinMaxScaler()
pjm['PJME_MW_scaled'] = scaler.fit_transform(pjm[['PJME_MW']])

# Save scaler (needed by the web app to convert predictions back to MW)
with open(os.path.join(MODELS, 'pjm_scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
print('Scaler saved to models/pjm_scaler.pkl')

# Save processed CSV (needed by the web app to look up 24-hour input windows)
pjm.to_csv(os.path.join(PROC, 'pjm_processed.csv'))
print(f'Processed data saved to data/processed/pjm_processed.csv  ({pjm.shape})')
pjm[['PJME_MW', 'PJME_MW_scaled']].tail(3)

In [ ]:
# ── Create 24-hour sliding window sequences ───────────────────────────────────
vals = pjm['PJME_MW_scaled'].values
X, y = [], []
for i in range(WINDOW, len(vals)):
    X.append(vals[i - WINDOW : i])
    y.append(vals[i])

X = np.array(X)   # shape: (n_samples, 24)
y = np.array(y)   # shape: (n_samples,)

# Chronological 80/20 split — no shuffling to prevent data leakage
split  = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'Training samples : {X_train.shape[0]:,}  ({pjm.index[WINDOW]} → {pjm.index[WINDOW + split - 1]})')
print(f'Test samples     : {X_test.shape[0]:,}  ({pjm.index[WINDOW + split]} → {pjm.index[-1]})')

In [ ]:
# ── Train Random Forest ───────────────────────────────────────────────────────
import time
print('Training Random Forest (n_estimators=100)...')
print('This takes 20–30 minutes. Please wait.')

t0 = time.time()
rf = RandomForestRegressor(
    n_estimators=100,
    n_jobs=-1,          # use all CPU cores
    random_state=42
)
rf.fit(X_train, y_train)
elapsed = time.time() - t0
print(f'Training complete in {elapsed/60:.1f} minutes.')

In [ ]:
# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred_scaled = rf.predict(X_test)

# Convert back to real MW values
y_pred_mw = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true_mw = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

rmse = np.sqrt(mean_squared_error(y_true_mw, y_pred_mw))
mae  = mean_absolute_error(y_true_mw, y_pred_mw)
mask = y_true_mw > 0
mape = np.mean(np.abs((y_true_mw[mask] - y_pred_mw[mask]) / y_true_mw[mask])) * 100
r2   = r2_score(y_true_mw, y_pred_mw)

print('=' * 45)
print('  Random Forest — PJM East Results')
print('=' * 45)
print(f'  RMSE  : {rmse:,.2f} MW')
print(f'  MAE   : {mae:,.2f} MW')
print(f'  MAPE  : {mape:.4f} %')
print(f'  R²    : {r2:.4f}')
print('=' * 45)

In [ ]:
# ── Save model ────────────────────────────────────────────────────────────────
model_path = os.path.join(MODELS, 'rf_pjm.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(rf, f)

size_mb = os.path.getsize(model_path) / 1_000_000
print(f'Model saved: {model_path}')
print(f'File size  : {size_mb:.1f} MB')
print()
print('Files ready for the web app:')
print('  models/rf_pjm.pkl              — trained Random Forest model')
print('  models/pjm_scaler.pkl          — scaler to convert predictions to MW')
print('  data/processed/pjm_processed.csv — processed data for input lookup')
print()
print('Run the web app:')
print('  cd predict_app')
print('  streamlit run app.py')